<a href="https://colab.research.google.com/github/bsudharsh1979/NATcolab/blob/main/notebooks/2_ALM_Config_Understanding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**NATcolab** · NVIDIA NeMo Agent Toolkit · *Asset Lifecycle Management with Agentic AI* — Google Colab edition (hosted NVIDIA NIMs)

# Asset Lifecycle Management - Configuration & Tool Customization

> **🟢 Colab edition.** This is the Google Colab adaptation of the NVIDIA DLI lab notebook. Instead of the lab's Docker containers it uses:
> - **Hosted NVIDIA NIMs** on build.nvidia.com — `nvidia/nemotron-3-ultra-550b-a55b` (planning, tool calling, text-to-SQL), `nvidia/nemotron-3.5-lightning-30b-a3b` (code generation), `nvidia/nemotron-3-nano-omni-30b-a3b-reasoning` (multimodal judge), `nvidia/nemotron-3.5-embed-1b` (embeddings)
> - **MOMENT-1-small**, an open-source time-series foundation model that runs locally, in place of the self-hosted NV-Tesseract NIM
> - a local Python **code sandbox** and a **Phoenix** server started from the notebook
>
> **Before you run:** add your NVIDIA API key (get one at [build.nvidia.com](https://build.nvidia.com)) as a Colab secret named `NVIDIA_API_KEY` — 🔑 icon in the left sidebar → *Add new secret* → enable *Notebook access*. A CPU runtime is enough.
> Colab gives every notebook its own VM, so run the **setup cell below at the top of every notebook** (≈5–8 min on a fresh runtime, seconds afterwards). Paths that were `/dli/task/...` in the lab are `/content/NATcolab/...` here.

> **Prerequisites:** none in Colab — the setup cell prepares everything and the cells below (re)start the backend.

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Modify LLM parameters** (temperature) to control response behavior
2. **Update existing tools** to add new features (color preferences in plots)
3. **Create custom tools** from scratch (action recommendation)
4. **Register tools** in the NAT configuration
5. **Update system prompts** to guide tool usage
6. **Test and validate** configuration changes
7. **Understand the NAT configuration file** structure

In [ ]:
# ⚙️ NATcolab setup — run first (≈5–8 min on a fresh runtime; a few seconds if already done in this session)
# Clones github.com/bsudharsh1979/NATcolab, creates the NAT virtual env (nvidia-nat 1.2.1 + ALM agent),
# builds the NASA turbofan database, trains the RUL model and checks the hosted NVIDIA NIMs.
import os, sys, subprocess
ALM_HOME = os.environ.get("ALM_HOME", "/content/NATcolab")
if not os.path.exists(f"{ALM_HOME}/colab_bootstrap.py"):
    subprocess.run(["git", "clone", "-q", "https://github.com/bsudharsh1979/NATcolab.git", ALM_HOME], check=True)
sys.path.insert(0, ALM_HOME)
os.chdir(ALM_HOME)

import colab_bootstrap
colab_bootstrap.bootstrap()

In [ ]:
from nat_client import NATClient, link
client = NATClient()

> **Reset codebase:** If you need to restore all source files and config to their original state (e.g. after testing a solution), run the cell below.

In [ ]:
client.reset()

---
# Understanding NAT Configuration

## What is NAT?

**NVIDIA Agent Toolkit (NAT)** is a configuration-driven framework for building AI agents. Instead of writing agent orchestration code from scratch, you describe *what* your agent should do in a YAML config file and NAT handles the rest — LLM calls, tool execution, reasoning loops, and response formatting.

The core idea: **one config file = one fully functional agent.**

## How NAT Works

```
  config-reasoning.yml
  ┌─────────────────────────┐
  │  llms:                  │ ── Which LLMs to wire up
  │  functions:             │ ── Which tools to register
  │  agents:                │ ── Agent type + tool set + prompt
  │  telemetry / evaluation │ ── Observability & eval settings
  └────────────┬────────────┘
               │
        NAT reads the config
        and builds the workflow
               │
               ▼
  ┌─────────────────────────┐
  │  Runtime Workflow Graph │  LLMs, tools, and agents
  │  (generated at startup) │  wired together automatically
  └────────────┬────────────┘
               │
   User Query ─┤
               ▼
       Agent executes a
       ReAct reasoning loop
       using the wired tools
               │
               ▼
        Final Response
```

The key insight is that **NAT generates the entire agentic workflow graph at startup from the config file**. You don't write orchestration code — you declare the LLMs, tools, and agent behavior in YAML, and NAT wires them into a working graph. Change the config, restart, and you get a different agent with different capabilities.

The agent itself uses a **ReAct (Reasoning + Acting)** loop — it thinks, calls a tool, observes the result, and repeats until it can answer — but you control *what* it can reason over and *which* tools it can call entirely through the config.

## Key CLI Commands

| Command | What it does |
|---|---|
| `nat serve --config config-reasoning.yml` | Starts the agent as a persistent backend API (port 8000). Accepts queries over HTTP. |
| `nat run --config config-reasoning.yml --input "your query"` | Runs a single query through the agent and exits — no server needed. Useful for quick testing. |

> **In this course** you don't need to run these manually — `NATClient` handles `nat serve` for you in the background. Each time you call `client.start()` or `client.restart()`, it launches/reloads the backend automatically.

## The config-reasoning.yml File

This single YAML file is the source of truth for your agent. It controls:

| Section | Purpose |
|---|---|
| **`llms`** | Which models to use (reasoning LLM, coding LLM, analyst LLM) |
| **`embedders`** | Vector embedding models for RAG/retrieval |
| **`functions`** | Available tools — each maps a name to a tool class and its settings |
| **`agents`** | Agent type, which tools it can access, and its system prompt |
| **`telemetry`** | Observability settings (Phoenix tracing) |
| **`evaluation`** | Eval dataset path and scoring settings |

The workflow throughout this notebook is simple: **edit the config → restart the backend → test with a query**.

---
# Exercise 2.1: Temperature Parameter Configuration

## Objective

Modify the LLM temperature parameter and observe how it affects response behavior.

## Background: What is Temperature?

**Temperature** controls the randomness of LLM outputs:

- **Low temperature (0.0-0.3)**: 
  - More deterministic and focused
  - Chooses most probable tokens
  - Best for factual, consistent responses
  - Example: "The RUL is 42 cycles."

- **Medium temperature (0.5-0.7)**: 
  - Balanced creativity and consistency
  - Good for general-purpose tasks
  - Example: "The remaining useful life is approximately 42 cycles, suggesting maintenance should be scheduled soon."

- **High temperature (0.8-1.0)**: 
  - More creative and diverse
  - Explores less probable tokens
  - Can be verbose or less focused
  - Example: "Based on the analysis of sensor patterns and degradation trends, the engine exhibits characteristics suggesting approximately 42 operational cycles remain before maintenance intervention becomes critical..."


## Tasks
### Step 1: Modify the Configuration

1. Open `/content/NATcolab/configs/config-reasoning.yml`
2. Find the `reasoning_llm` section (in the `llms:` section)
3. Add the `temperature` parameter:

```yaml
reasoning_llm:
  _type: nim
  model_name: "nvidia/nemotron-3-ultra-550b-a55b"
  max_tokens: 8192
  temperature: 0.1  # ← ADD THIS LINE (try 0.1, then 0.5, then 1.0)
```

> **Note:** The default temperature when unspecified is `0.0`. Experiment with values
> like `0.1`, `0.5`, and `1.0` to see how the response style changes.


> **Editing files in Colab:** open the 📁 file browser (left sidebar) → `NATcolab/configs/config-reasoning.yml` → double-click to open it in Colab's editor, edit, and press `Ctrl+S`.

<details>
<summary><b>📋 Solution (Colab): apply the edits programmatically</b></summary>

The DLI lab copies pre-edited files from `/dli/task/Exercise_Solutions/`. In Colab, run the solution cell below — `exercise_solutions.exercise_1_temperature()` applies exactly the edits described above (idempotent).

> **To restore original files after testing:** run `client.reset()` — this reverts all source files and config back to the base state.
</details>


### Step 2: Start the Backend

After saving your config edit, use the cells below to restart the backend and test your change.

> **First time only (backend not yet running):** use `client.start()`.  
> **After a config change (backend already running):** use `client.restart()`.

### Step 3: Test with the Same Query

Run the test cell below at each temperature value (0.1, 0.5, 1.0) and note how the response style changes.

**Document your observations:**

| Temperature | Response Length | Detail Level | Consistency | Best For |
|-------------|----------------|--------------|-------------|----------|
| 0.1         |                |              |             |          |
| 0.5         |                |              |             |          |
| 1.0         |                |              |             |          |

### Expected Results

- **Temperature 0.1**: Concise, factual, consistent responses
- **Temperature 0.5**: Balanced detail and conciseness
- **Temperature 1.0**: More verbose, creative, potentially less focused

**Solution Reference:** See `/content/NATcolab/Exercise_Solutions/Exercise_1_Temperature/config-reasoning.yml`

In [ ]:
# Run the same query at each temperature value and compare the responses.
# (Each iteration applies the config edit with the solution helper and restarts the backend.)
import exercise_solutions as sol

temperature_responses = {}
for t in (0.1, 0.5, 1.0):
    sol.exercise_1_temperature(t)
    client.restart()
    temperature_responses[t] = client.query("What is predictive maintenance and why is it important?")

In [ ]:
for t, r in temperature_responses.items():
    print(f"temperature={t}: {len(r.split())} words")

> ✅ Once you've tested all three temperatures, proceed to **Exercise 2.2** below.

---
# Exercise 2.2: Plotting Tool Enhancement

## Objective

Modify the plotting tool to accept user-specified colors for line charts.

## Background

Currently, the `plot_line_chart_tool` uses a default blue color for all plots. We want to:
1. Accept a `color` parameter from the user
2. Use the specified color if provided
3. Fall back to default if not specified
4. Update the system prompt to tell the agent about this capability

## Files to Modify

1. `plot_line_chart_tool.py` - Add color parameter
2. `plot_utils.py` - Implement color handling
3. `config-reasoning.yml` - Update prompts

## Tasks

### Step 1: Modify the plotting tool

#### 1a. Edit `plot_line_chart_tool.py`

Open: `/content/NATcolab/asset_lifecycle_management_agent/src/asset_lifecycle_management_agent/plotting/plot_line_chart_tool.py`

**Change 1 — Add `color` to the input schema** (find the `PlotLineChartInputSchema` class):

```python
class PlotLineChartInputSchema(BaseModel):
    data_json_path: str = Field(...)
    x_axis_column: str = Field(...)
    y_axis_column: str = Field(...)
    plot_title: str = Field(...)
    color: str = Field(description="The color for the line chart", default="blue")  # ← ADD
```

**Change 2 — Accept `color` in the response function** (find `_response_fn`):

```python
async def _response_fn(data_json_path: str, x_axis_column: str,
                       y_axis_column: str, plot_title: str,
                       color: str = "blue") -> str:  # ← ADD color parameter
```

**Change 3 — Pass `color` to `create_line_chart`** (find the call inside `_response_fn`):

```python
html_filepath, png_filepath = create_line_chart(
    output_dir=config.output_folder,
    data_json_path=data_json_path,
    x_col=x_axis_column,
    y_col=y_axis_column,
    title=plot_title,
    color=color  # ← ADD
)
```

#### 1b. Edit `plot_utils.py`

Open: `/content/NATcolab/asset_lifecycle_management_agent/src/asset_lifecycle_management_agent/plotting/plot_utils.py`

**Change 4 — Accept `color` in `create_line_chart`** (find the function signature):

```python
def create_line_chart(output_dir: str, data_json_path: str, x_col: str,
                      y_col: str, title: str,
                      color: str = "blue") -> Tuple[str, Optional[str]]:  # ← ADD
```

**Change 5 — Use the color** (find the `line=dict(...)` and `marker=dict(...)` lines inside the Plotly trace):

```python
line=dict(color=color, width=3),        # was '#1f77b4'
marker=dict(size=6, color=color),       # was '#1f77b4'
```

### Step 2: Update the system prompt in config

In `configs/config-reasoning.yml`, find the `system_prompt` under `data_analysis_assistant` and add the following line inside the **Analysis and Plotting Tools** section:

```
- plot_line_chart: ... Extract color preference from user prompt (e.g., "plot in blue", "use red color") and pass as color parameter.
```

<details>
<summary><b>📋 Solution (Colab): apply the edits programmatically</b></summary>

The DLI lab copies pre-edited files from `/dli/task/Exercise_Solutions/`. In Colab, run the solution cell below — `exercise_solutions.exercise_2_plot_color()` applies exactly the edits described above (idempotent).

> **To restore original files after testing:** run `client.reset()` — this reverts all source files and config back to the base state.
</details>

In [ ]:
import exercise_solutions as sol
sol.exercise_2_plot_color()

### Step 3: Restart Backend

In [ ]:
client.restart()   # reloads the updated config-reasoning.yml

In [ ]:
client.check_health()

### Step 4: Test Color Preferences

Run the cell below to test with two queries — one with the default color and one with a specified color.

> **Remember:** Save each file (`Ctrl+S`) after making edits in the Colab editor.

In [ ]:
# Query 1: Default color (should be blue)
response1 = client.query(
    "Show me a line chart of sensor_measurement_4 for unit 69 in FD001 training data over time"
)

In [ ]:
# Visualize the plot
client.show("line_chart_time_in_cycles_vs_sensor_measurement_4.png")

In [ ]:
# Query 2: Explicit red color
response2 = client.query(
    "Plot sensor_measurement_4 vs time_in_cycles for unit 69 in FD001 using red color"
)

In [ ]:
# Visualize the plot (should now be red)
client.show("line_chart_time_in_cycles_vs_sensor_measurement_4.png")

---
# Exercise 2.3: Action Recommendation Tool

## Objective

Create a new tool that generates actionable maintenance recommendations based on analysis results.

## Background

Currently, the agent can:
- Retrieve data
- Detect anomalies
- Predict RUL
- Create visualizations

But it doesn't provide **actionable recommendations** like:
- "Schedule inspection within 10 cycles"
- "Monitor sensor 4 closely for next 20 cycles"
- "Consider preventive maintenance for this unit"

We'll create an `action_recommendation_tool` that takes analysis results and suggests actions.

## Files to Create/Modify

1. Create: `recommenders/action_recommendation_tool.py` - New tool implementation
2. Modify: `register.py` - Register the new tool
3. Modify: `config-reasoning.yml` - Add tool to configuration

## Tasks

### Step 1: Add the new Action recommendation tool

#### 1a: Create the Tool File

Create the following file (you may need to create the `recommenders/` directory first):

```
asset_lifecycle_management_agent/src/asset_lifecycle_management_agent/recommenders/
├── __init__.py                      # empty file
└── action_recommendation_tool.py    # the new tool
```

The tool follows the standard NAT pattern you've already seen in other tools:

1. **Config class** — Extends `FunctionBaseConfig` with `name="action_recommendation_tool"`. Needs one field: `llm_name` (which LLM to use for generating recommendations).

2. **Input schema** — A Pydantic `BaseModel` with two fields:
   - `analysis_result: str` — The analysis output to base recommendations on
   - `context: str` — Additional context (engine unit, dataset, etc.)

3. **Response function** — Builds a prompt with the analysis result and context, calls the LLM via `builder.get_llm(config.llm_name)`, and formats the output as a numbered list of recommendations with urgency levels (URGENT / HIGH / MEDIUM / LOW).

4. **Registration** — Decorated with `@register_function(config_type=..., framework_wrappers=[LLMFrameworkEnum.LANGCHAIN])`

Use the existing tools (e.g., `predict_rul_tool.py`) as a reference for the boilerplate structure.

#### 1b: Register the Tool

Add one import line to `register.py`:

```python
from .recommenders import action_recommendation_tool
```

#### 1c: Update config file

In `configs/config-reasoning.yml`, make two additions:

**a) Under `functions:`, add the tool config:**

```yaml
  action_recommendation:
    _type: action_recommendation_tool
    llm_name: reasoning_llm
```

**b) Under `data_analysis_assistant.tool_names`, add it to the list:**

```yaml
    tool_names: [
      sql_retriever,
      predict_rul,
      ...,
      action_recommendation   # ← ADD
    ]
```

**c) Optionally, add a usage hint to the `data_analysis_assistant.system_prompt`:**

```yaml
      6. **Action Recommendation Tool**
         - Use action_recommendation to provide actionable maintenance recommendations
           based on any analysis results.
         - Call this tool after completing analysis to give users specific next steps.
```

<details>
<summary><b>📋 Solution (Colab): apply the edits programmatically</b></summary>

The DLI lab copies pre-edited files from `/dli/task/Exercise_Solutions/`. In Colab, run the solution cell below — `exercise_solutions.exercise_3_action_recommendation()` applies exactly the edits described above (idempotent).

> **To restore original files after testing:** run `client.reset()` — this reverts all source files and config back to the base state.
</details>

In [ ]:
import exercise_solutions as sol
sol.exercise_3_action_recommendation()
print(open(f"{ALM_HOME}/asset_lifecycle_management_agent/src/asset_lifecycle_management_agent/recommenders/action_recommendation_tool.py").read())

### Step 2: Reinstall the package for NAT to recognize the new tool

In [ ]:
# Reinstall to pick up new tool
!cd {ALM_HOME}/asset_lifecycle_management_agent && uv pip install --python {client.python_bin} -q -e .

### Step 3: Restart Backend

In [ ]:
client.restart()   # reloads the updated config-reasoning.yml

In [ ]:
client.check_health()

### Step 4: Test Action Recommendations

Run the cell below to test two recommendation workflows — one driven by RUL prediction and one by anomaly detection.

In [ ]:
# Query 1: RUL prediction + recommendations
response1 = client.query(
    "Predict the remaining useful life for unit 15 in FD001 and recommend maintenance actions",
    timeout=900,
)

In [ ]:
# Query 2: Ground truth RUL + recommendations
response2 = client.query(
    "What is the ground truth RUL for unit 20 in FD001? Based on that, provide maintenance recommendations",
    timeout=900,
)

---
# Key Takeaways

In this notebook, you learned:

1. ✅ How to modify LLM parameters (temperature) to control behavior
2. ✅ How to extend existing tools with new features (color support)
3. ✅ How to create custom tools from scratch (action recommendations)
4. ✅ The structure of NAT configuration files
5. ✅ How to register and integrate new tools
6. ✅ How to update system prompts to guide tool usage
7. ✅ The workflow for testing configuration changes

## Configuration Best Practices

- **Always backup** before making changes
- **Test incrementally**: Change one thing at a time
- **Document changes**: Note what worked and what didn't
- **Version your configs**: Keep different versions for different use cases
- **Update prompts**: Always update system prompts when adding tools

## What's Next?

In **Notebook 3**, you will:
- Set up observability with Phoenix
- Enable tracing and monitoring
- Run complex queries and analyze their execution
- Learn to debug agent behavior using traces
- Understand token usage and latency optimization

**[Continue to Notebook 3 - Observability →](3_ALM_Observability.ipynb)**